# Visualisation du dataset

1. Exemples d'images, avec les boîtes des deux coins et le nom de chaque carte (vert : coin visible, rouge : coin trop caché)
2. Les déformations, une par une : ce que fait chaque étape de `scene.py`
3. Statistiques : cartes par image, apparitions de chaque carte, tailles et visibilité des coins

Prérequis : `python -m cards.dataset.assets` puis `python -m cards.dataset.generate_dataset`.

In [ ]:
import os
import sys
while not os.path.isdir(".git"):  # remonte jusqu'à la racine du repo, pour que le chemin data/ fonctionne
    os.chdir("..")
sys.path.insert(0, os.getcwd())  # pour pouvoir importer le package cards

import json
from collections import Counter

import matplotlib.patches as patches
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

DATA_DIR = "data/synthetic"
SPLIT = "train"
annotations = json.load(open(f"{DATA_DIR}/{SPLIT}/annotations.json"))
items = annotations["items"]
n_cards = sum(len(item["cards"]) for item in items)
n_visible = sum(corner["visible"] for item in items for card in item["cards"] for corner in card["corners"])
print(f"{len(items)} images, {n_cards} cartes, {n_visible} coins visibles sur {2 * n_cards}")

## 1. Exemples

In [ ]:
def show(ax, item):
    ax.imshow(Image.open(f"{DATA_DIR}/{SPLIT}/{item['image']}"))
    for card in item["cards"]:
        for corner in card["corners"]:
            x1, y1, x2, y2 = corner["box"]
            color = "lime" if corner["visible"] else "red"
            ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color, linewidth=1.5))
        # Le nom est écrit au-dessus du premier coin visible
        x1, y1, _, _ = next((c["box"] for c in card["corners"] if c["visible"]), card["corners"][0]["box"])
        ax.text(x1, y1 - 3, card["name"], color="black", fontsize=6, backgroundcolor="lime")
    ax.set_title(item["image"], fontsize=9)
    ax.axis("off")

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for ax, item in zip(axes.flat, items):
    show(ax, item)
plt.tight_layout()

## 2. Les déformations, une par une

On rejoue ici les étapes de [scene.py](scene.py) séparément, avec les mêmes fonctions que la génération, pour voir l'effet de chacune.

### 2.1 Une carte, de l'image d'origine à ses coins

Le fond gris montre les zones transparentes. En couleur, le masque des deux coins (rang + enseigne) : il subit exactement les mêmes transformations que la carte. À la dernière étape, les cadres verts sont les boîtes des coins enregistrées.

In [ ]:
import numpy as np
from PIL import ImageEnhance, ImageFilter

from cards.deck import DECK
from cards.dataset.scene import (
    MAX_CARD_HEIGHT, MIN_CARD_HEIGHT, corner_mask, load_card_images, perspective, place_cards, random_background,
)

card_images = load_card_images()
card = next(c for c in DECK if c.name == "king of spades")
rng = np.random.default_rng(3)

original = card_images[card]
mask = corner_mask(original.size)
size = (round(original.width * 150 / original.height), 150)
resized, mask_resized = original.resize(size, Image.LANCZOS), mask.resize(size, Image.NEAREST)
warped, mask_warped = perspective(resized, rng, 0.15, mask_resized)
rotated = warped.rotate(15, resample=Image.BICUBIC, expand=True)
mask_rotated = mask_warped.rotate(15, resample=Image.NEAREST, expand=True)
bbox = rotated.getchannel("A").getbbox()
cropped, mask_cropped = rotated.crop(bbox), mask_rotated.crop(bbox)

def show_sprite(ax, sprite, title, mask=None, boxes=False):
    ax.set_facecolor("lightgray")
    ax.imshow(sprite)
    if mask is not None:
        m = np.asarray(mask)
        ax.imshow(np.ma.masked_equal(m, 0), cmap="cool", alpha=0.5, vmin=1, vmax=2)
    if boxes:
        for c in (1, 2):
            ys, xs = np.nonzero(m == c)
            ax.add_patch(patches.Rectangle((xs.min() - 0.5, ys.min() - 0.5), xs.max() - xs.min() + 1, ys.max() - ys.min() + 1,
                                           fill=False, edgecolor="lime", linewidth=2))
    ax.add_patch(patches.Rectangle((-0.5, -0.5), sprite.width, sprite.height, fill=False, edgecolor="red", linestyle="--"))
    ax.set_xlim(-10, max(sprite.width, 160) + 10)
    ax.set_ylim(max(sprite.height, 190) + 10, -10)
    ax.set_title(title, fontsize=9)
    ax.axis("off")

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
axes[0].set_facecolor("lightgray"); axes[0].imshow(original)
axes[0].imshow(np.ma.masked_equal(np.asarray(mask), 0), cmap="cool", alpha=0.5, vmin=1, vmax=2)
axes[0].set_title(f"1. image d'origine ({original.width}×{original.height})", fontsize=9); axes[0].axis("off")
show_sprite(axes[1], resized, "2. redimensionnée à 150 px de haut", mask_resized)
show_sprite(axes[2], warped, "3. perspective", mask_warped)
show_sprite(axes[3], rotated, "4. rotation de 15° (canevas agrandi)", mask_rotated)
show_sprite(axes[4], cropped, "5. découpe au ras, boîtes des coins", mask_cropped, boxes=True)
plt.tight_layout()

### 2.2 La perspective

Chaque coin est rapproché du centre de 0 à 15 % de la largeur et de la hauteur, au hasard : la carte semble vue en biais. Huit tirages sur la même carte :

In [ ]:
rng = np.random.default_rng(0)
fig, axes = plt.subplots(1, 8, figsize=(18, 3))
for ax in axes:
    show_sprite(ax, perspective(resized, rng, 0.15), "")
plt.tight_layout()

### 2.3 La taille des cartes et les recouvrements

Une hauteur est tirée par image, entre 75 px et un maximum qui dépend du nombre de cartes. Ligne du haut : la taille minimale ; ligne du bas : la taille maximale, pour 1 à 5 cartes. Les cartes peuvent se recouvrir, tant que chacune garde au moins un coin visible.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7.5))
for col, n in enumerate(range(1, 6)):
    for row, height in enumerate([MIN_CARD_HEIGHT, MAX_CARD_HEIGHT[n]]):
        rng = np.random.default_rng(n)
        scene = Image.new("RGB", (384, 384), (40, 110, 70))
        place_cards(scene, card_images, rng, n, height, max_rotation=20, max_perspective=0.15)
        axes[row, col].imshow(scene)
        axes[row, col].set_title(f"{n} carte(s), hauteur {height} px", fontsize=9)
        axes[row, col].axis("off")
plt.tight_layout()

### 2.4 Les trois types de fond

Couleur unie, dégradé (vertical ou horizontal) ou taches de couleur floues, avec une chance sur trois chacun. Deux exemples de chaque :

In [ ]:
# random_background() tire d'abord le type de fond : on cherche des graines qui donnent chaque type
names = {0: "couleur unie", 1: "dégradé", 2: "taches floues"}
fig, axes = plt.subplots(1, 6, figsize=(18, 3.2))
ax_iter = iter(axes)
for kind in range(3):
    seeds = [s for s in range(100) if np.random.default_rng(s).integers(3) == kind][:2]
    for seed in seeds:
        ax = next(ax_iter)
        ax.imshow(random_background(np.random.default_rng(seed), 384))
        ax.set_title(names[kind], fontsize=9)
        ax.axis("off")
plt.tight_layout()

### 2.5 Les retouches de l'image entière

Appliquées après avoir posé les cartes : luminosité et contraste (facteur entre 0,7 et 1,3), flou (sur 30 % des images, rayon 0,3 à 1,2 px), puis grain (bruit d'écart-type 6). Chaque retouche est montrée seule, à ses valeurs extrêmes. Le grain est montré en zoom.

In [ ]:
rng = np.random.default_rng(7)
base = random_background(rng, 384)
place_cards(base, card_images, rng, 2, 195, max_rotation=20, max_perspective=0.15)

def add_grain(image, sigma=6):
    array = np.asarray(image, dtype=np.float32) + np.random.default_rng(0).normal(0, sigma, (image.height, image.width, 3))
    return Image.fromarray(np.clip(array, 0, 255).astype(np.uint8))

zoom = lambda image: image.crop((160, 160, 224, 224)).resize((384, 384), Image.NEAREST)
variants = [
    (base, "sans retouche"),
    (ImageEnhance.Brightness(base).enhance(0.7), "luminosité × 0,7"),
    (ImageEnhance.Brightness(base).enhance(1.3), "luminosité × 1,3"),
    (ImageEnhance.Contrast(base).enhance(0.7), "contraste × 0,7"),
    (ImageEnhance.Contrast(base).enhance(1.3), "contraste × 1,3"),
    (base.filter(ImageFilter.GaussianBlur(1.2)), "flou, rayon 1,2"),
    (zoom(base), "zoom, sans grain"),
    (zoom(add_grain(base)), "zoom, avec grain"),
]
fig, axes = plt.subplots(2, 4, figsize=(16, 8.5))
for ax, (image, title) in zip(axes.flat, variants):
    ax.imshow(image)
    ax.set_title(title, fontsize=10)
    ax.axis("off")
plt.tight_layout()

## 3. Statistiques

Les cartes sont tirées au hasard, donc leur nombre d'apparitions varie autour de la moyenne. L'écart-type attendu par le seul hasard vaut environ √(moyenne) : il devient négligeable en proportion quand le dataset grandit.

In [ ]:
cards_per_image = Counter(len(item["cards"]) for item in items)
card_counts = Counter(card["name"] for item in items for card in item["cards"])
corners = [corner for item in items for card in item["cards"] for corner in card["corners"]]
corner_heights = [c["box"][3] - c["box"][1] for c in corners]
visibilities = [c["visibility"] for c in corners]

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
axes[0].bar([str(k) for k in sorted(cards_per_image)], [cards_per_image[k] for k in sorted(cards_per_image)])
axes[0].set_title("Cartes par image")
axes[1].bar(range(len(card_counts)), sorted(card_counts.values()))
axes[1].set_title("Apparitions de chaque carte (triées)")
axes[2].hist(corner_heights, bins=30)
axes[2].set_title("Hauteur des boîtes de coin (pixels)")
axes[3].hist(visibilities, bins=20)
axes[3].set_yscale("log")
axes[3].set_title("Visibilité des coins (échelle log)")
plt.tight_layout()

mean = sum(card_counts.values()) / 52
print(f"Apparitions par carte : min {min(card_counts.values())}, max {max(card_counts.values())}, moyenne {mean:.1f} (52 cartes : {len(card_counts) == 52})")
print(f"Écart-type observé : {np.std(list(card_counts.values())):.1f} ; attendu par le hasard : {np.sqrt(mean):.1f}")